# Session 2.3 : Relational Data

_Analytics Through Coding Autumn 2026_

---

It is rare that an analysis uses only one table.

Information is often split across several related datasets:

* one table contains the observations we want to analyse;
* other tables contain additional information about those observations.

The analytical task is not simply to **join tables**. We also need to check that the join produced the result we expected.

In this session we will focus on:

* identifying the variables that link tables;
* understanding one-to-many relationships;
* combining tables with `merge()`;
* checking whether joins changed the number of observations;
* identifying records that did not match;
* using semi-join and anti-join logic;
* combining several related datasets.

---

## Starting out

We will continue using the `nycflights13` datasets.

The main `flights` table records individual flights. Other tables contain information about airlines, airports, aircraft and weather.

In [ ]:
import pandas as pd
import numpy as np

flights = pd.read_csv("../Data/nycflights13_flights.csv", index_col=0)
flights.reset_index(drop=True, inplace=True)

airlines = pd.read_csv("../Data/nycflights13_airlines.csv")
airports = pd.read_csv("../Data/nycflights13_airports.csv")
planes = pd.read_csv("../Data/nycflights13_planes.csv")
weather = pd.read_csv("../Data/nycflights13_weather.csv")

print("flights:", flights.shape)
print("airlines:", airlines.shape)
print("airports:", airports.shape)
print("planes:", planes.shape)
print("weather:", weather.shape)

## How are the tables related?

![GitHub Codespaces](fligths_data.png)

Before joining anything, identify:

1. **What does one row represent in each table?**
2. **Which variable links the tables?**
3. **Is the linking variable unique in one of the tables?**

For example:

* one row in `flights` = one flight;
* one row in `airlines` = one airline;
* `carrier` links the two tables.

Many flights can belong to the same airline, so this is a **many-to-one** relationship from `flights` to `airlines`.



A useful check is whether the lookup key is unique in the table that is supposed to contain one row per entity.

<div class="alert alert-warning">
<b>Why does uniqueness matter?</b>

If we expect one airline record for each carrier but the airline table contains duplicate carrier codes, a join could unexpectedly duplicate rows in the flights table.

Always understand the relationship between the join keys before merging.
</div>

## Question 1: What is the full airline name for each flight?

The `flights` table only contains short carrier codes such as `UA`, `DL` and `B6`.

The `airlines` table contains the corresponding airline names.

For this question, we want to keep **every flight** and add airline information where a match exists.

That makes a **left join** the natural choice.

The argument:

```python
validate="many_to_one"
```

asks pandas to confirm that many rows in `flights` are allowed to match one row in `airlines`.

If the relationship is not what we expected, pandas will raise an error rather than silently producing a potentially incorrect dataset.

## Validate the join

A join is not finished simply because the code ran.

Before the join:

> one row = one flight

After adding airline information:

> one row should still = one flight

Therefore the number of rows should not change.

We should also check whether any flights failed to match an airline.

### Exercise 1 — Join and validate

Join `flights` to `airlines` using a left join.

Store the result in `flight_airline_check`.

Then:

1. Use `validate="many_to_one"`.
2. Confirm that the number of rows has not changed.
3. Count the number of missing airline names.
4. Display `carrier`, `name`, `origin` and `dest` for the first 10 rows.

## Inner, left, right and outer joins

Pandas supports several join types.

| Join | What it keeps |
|---|---|
| `inner` | Only rows with a match in both tables |
| `left` | All rows from the left table, plus matching information from the right |
| `right` | All rows from the right table, plus matching information from the left |
| `outer` | All rows from both tables |

![GitHub Codespaces](mutating_joins.png)

For analytical work, the important question is not:

> Which join type do I know?

It is:

> **Which observations should remain in the resulting dataset?**

For example, compare an inner and left join between flights and airlines.

If every carrier in `flights` is present in `airlines`, the inner and left joins will have the same number of rows.

If some carriers are missing from the lookup table, an inner join would silently remove those flights.

That is why validation matters.

## Finding unmatched records with `indicator=True`

Pandas can tell us whether each observation matched using the `indicator` argument.

The `_merge` column contains:

* `both` — a match was found;
* `left_only` — the observation existed only in the left table;
* `right_only` — the observation existed only in the right table.

This is especially useful when we need to investigate failed matches.

## Question 2: Which destinations cannot be matched to the airport table?

The destination variable is called `dest` in `flights`, but the airport code is called `faa` in `airports`.

The variable names do not have to be identical.

We specify the matching columns explicitly.

Now we can isolate destination codes that did not match an airport record.

This is a much more useful check than simply noticing missing airport names later.

A failed match could mean:

* the lookup table does not cover every destination;
* the join key uses a different coding convention;
* the source data contain an error;
* the wrong join key was selected.

Do not automatically treat an unmatched value as bad data.

### Exercise 2 — Airport join

Add destination airport information to the `flights` table.

1. Match `flights["dest"]` to `airports["faa"]`.
2. Use a left join.
3. Validate the relationship as `many_to_one`.
4. Use `indicator=True`.
5. Count how many flight records matched and did not match.
6. List the unique unmatched destination codes.

## Filtering joins: semi and anti join logic

Sometimes we do not want to add columns from another table.

Instead, we only want to know whether a match exists.

Two useful concepts are:

**Semi join**

> Keep rows from the left table where a match exists in the right table.

**Anti join**

> Keep rows from the left table where no match exists in the right table.

Pandas does not have dedicated `semi_join()` or `anti_join()` functions, but the logic is easy to implement.

### Semi join example

Keep only flights whose destination appears in the airport table.

### Anti join example

Keep only flights whose destination does **not** appear in the airport table.

The anti join is particularly useful for **data validation** because it identifies records that failed to find a match.

### Exercise 3 — Anti join

Using `flights` and `planes`, identify flights whose `tailnum` does not appear in the aircraft table.

1. Ignore flights where `tailnum` itself is missing.
2. Keep only unmatched flights.
3. Display the unique unmatched tail numbers.
4. Count how many flight records have an unmatched non-missing tail number.

## Question 3: Can we combine several lookup tables?

Yes. Joins can be chained.

For example, starting with one row per flight we can add:

* airline name using `carrier`;
* aircraft characteristics using `tailnum`;
* destination airport information using `dest` → `faa`.

The main rule is that we should validate each relationship rather than blindly joining tables together.

The number of columns increases because we have added information.

The number of rows should remain unchanged because one row should still represent one flight.

## Final Exercise — Build and validate an analytical dataset

Create a DataFrame called `flight_analysis` that contains:

* the original flight information;
* airline name;
* aircraft information;
* destination airport information.

Use left joins so that no flight is intentionally removed.

Then verify:

1. Each lookup relationship is treated as `many_to_one`.
2. The final number of rows equals the original number of flights.
3. Count missing airline names.
4. Count missing aircraft manufacturers.
5. Count missing destination airport names.

Finally display these columns for the first 10 rows:

* `carrier`
* airline `name`
* `tailnum`
* `manufacturer`
* `dest`
* destination airport name
* `arr_delay`

Because both airlines and airports contain a column called `name`, rename these clearly before joining.

## A join validation checklist

Before moving on from a merge, ask:

**1. What does one row represent before the join?**

**2. What should one row represent after the join?**

**3. Is the key unique where I expect it to be unique?**

**4. What relationship do I expect?**
* one-to-one
* one-to-many
* many-to-one
* many-to-many

**5. Did the number of rows change? Should it have?**

**6. Did any observations fail to match?**

**7. Did the join introduce unexpected missing values?**

A join that executes successfully is not necessarily a correct join.

## All Done!

In this session we worked with relational data and practised:

* identifying variables that connect datasets;
* recognising many-to-one relationships;
* combining datasets with `merge()`;
* choosing a join based on which observations should remain;
* using `validate=` to test the expected relationship;
* checking row counts after joining;
* using `indicator=True` to identify failed matches;
* implementing semi-join and anti-join logic;
* combining several lookup tables into one analytical dataset.

The key idea is:

> **Do not only ask whether the join ran. Ask whether the join produced the dataset you intended.**

We will now move from preparing and combining data to **exploratory analysis and visualisation**.